# 01 Data Understanding

## 1. Project Objective

This project examines pharmaceutical product-group sales recorded at hourly, daily, weekly, and monthly levels. Before cleaning or analysis begins, this notebook establishes what each source file contains, how the files differ, and whether the raw data has immediate quality issues.

The findings here define the data foundation for the later cleaning, EDA, time-series, SQL, Power BI, and business-insight work.

## 2. Notebook Setup

Only pandas is required at this stage. Visualization and numerical libraries will be introduced when the project reaches EDA.

In [1]:
import pandas as pd

## 3. Load the Raw Data


In [3]:
daily_df = pd.read_csv("../data/raw/salesdaily.csv")
hourly_df = pd.read_csv("../data/raw/saleshourly.csv")
weekly_df = pd.read_csv("../data/raw/salesweekly.csv")
monthly_df = pd.read_csv("../data/raw/salesmonthly.csv")

datasets = {
    "Daily": daily_df,
    "Hourly": hourly_df,
    "Weekly": weekly_df,
    "Monthly": monthly_df,
}

print("Loaded:", ", ".join(datasets))

Loaded: Daily, Hourly, Weekly, Monthly


## 4. Source File Inventory

The inventory confirms the number of records and fields available at each time grain. Previewing the first rows also helps verify that the files represent the same product groups.

In [5]:
summary_data = []

for name, df in datasets.items():
    rows = df.shape[0]
    columns = df.shape[1]

    summary_data.append({
        "Dataset": name,
        "Rows": rows,
        "Columns": columns
    })

dataset_summary = pd.DataFrame(summary_data)

dataset_summary

,Dataset,Rows,Columns
0,Daily,2106,13
1,Hourly,50532,13
2,Weekly,302,9
3,Monthly,70,9


In [13]:
for name, df in datasets.items():
    print(f"{name}: {df.columns.tolist()}")
    display(df.head())

Daily: ['datum', 'M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06', 'Year', 'Month', 'Hour', 'Weekday Name']


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,Year,Month,Hour,Weekday Name
0,1/2/2014,0.0,3.67,3.4,32.40,7.0,0.0,0.0,2.0,2014,1,248,Thursday
1,1/3/2014,8.0,4.00,4.4,50.60,16.0,0.0,20.0,4.0,2014,1,276,Friday
2,1/4/2014,2.0,1.00,6.5,61.85,10.0,0.0,9.0,1.0,2014,1,276,Saturday
3,1/5/2014,4.0,3.00,7.0,41.10,8.0,0.0,3.0,0.0,2014,1,276,Sunday
4,1/6/2014,5.0,1.00,4.5,21.70,16.0,2.0,6.0,2.0,2014,1,276,Monday


Hourly: ['datum', 'M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06', 'Year', 'Month', 'Hour', 'Weekday Name']


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,Year,Month,Hour,Weekday Name
0,1/2/2014 8:00,0.0,0.67,0.4,2.0,0.0,0.0,0.0,1.0,2014,1,8,Thursday
1,1/2/2014 9:00,0.0,0.00,1.0,0.0,2.0,0.0,0.0,0.0,2014,1,9,Thursday
2,1/2/2014 10:00,0.0,0.00,0.0,3.0,2.0,0.0,0.0,0.0,2014,1,10,Thursday
3,1/2/2014 11:00,0.0,0.00,0.0,2.0,1.0,0.0,0.0,0.0,2014,1,11,Thursday
4,1/2/2014 12:00,0.0,2.00,0.0,5.0,2.0,0.0,0.0,0.0,2014,1,12,Thursday


Weekly: ['datum', 'M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06']


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06
0,1/5/2014,14.00,11.67,21.3,185.95,41.0,0.0,32.0,7.0
1,1/12/2014,29.33,12.68,37.9,190.70,88.0,5.0,21.0,7.2
2,1/19/2014,30.67,26.34,45.9,218.40,80.0,8.0,29.0,12.0
3,1/26/2014,34.00,32.37,31.5,179.60,80.0,8.0,23.0,10.0
4,2/2/2014,31.02,23.35,20.7,159.88,84.0,12.0,29.0,12.0


Monthly: ['datum', 'M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06']


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06
0,2014-01-31,127.69,99.090,152.100,878.030,354.0,50.0,112.0,48.2
1,2014-02-28,133.32,126.050,177.000,1001.900,347.0,31.0,122.0,36.2
2,2014-03-31,137.44,92.950,147.655,779.275,232.0,20.0,112.0,85.4
3,2014-04-30,113.10,89.475,130.900,698.500,209.0,18.0,97.0,73.7
4,2014-05-31,101.79,119.933,132.100,628.780,270.0,23.0,107.0,123.7


**Note**
 _..._
 The availability of multiple time grains helps analyze pharma sales from different business angles: hourly data supports peak-demand analysis, daily and weekly data reveal operational sales patterns, and monthly data helps identify long-term trends and seasonality.

## 5. Schema and Data Types

The files share the `datum` field and eight medicine-category fields. Daily and hourly data also include calendar attributes. Reviewing `.info()` and a comparison table makes schema differences visible before cleaning.

In [23]:
for name, df in datasets.items():
    print(f"--- {name} dataset ---")
    df.info()
    print()

dtype_comparison = pd.DataFrame({
    name: df.dtypes.astype(str)
    for name, df in datasets.items()
})

dtype_comparison

--- Daily dataset ---
<class 'pandas.DataFrame'>
RangeIndex: 2106 entries, 0 to 2105
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   datum         2106 non-null   str    
 1   M01AB         2106 non-null   float64
 2   M01AE         2106 non-null   float64
 3   N02BA         2106 non-null   float64
 4   N02BE         2106 non-null   float64
 5   N05B          2106 non-null   float64
 6   N05C          2106 non-null   float64
 7   R03           2106 non-null   float64
 8   R06           2106 non-null   float64
 9   Year          2106 non-null   int64  
 10  Month         2106 non-null   int64  
 11  Hour          2106 non-null   int64  
 12  Weekday Name  2106 non-null   str    
dtypes: float64(8), int64(3), str(2)
memory usage: 247.1 KB

--- Hourly dataset ---
<class 'pandas.DataFrame'>
RangeIndex: 50532 entries, 0 to 50531
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------ 

,Daily,Hourly,Weekly,Monthly
Hour,int64,int64,NaN,NaN
M01AB,float64,float64,float64,float64
M01AE,float64,float64,float64,float64
Month,int64,int64,NaN,NaN
N02BA,float64,float64,float64,float64
N02BE,float64,float64,float64,float64
N05B,float64,float64,float64,float64
N05C,float64,float64,float64,float64
R03,float64,float64,float64,float64
R06,float64,float64,float64,float64


**Note...**
All datasets contain the `datum` date column and medicine category sales columns. Daily and hourly datasets also include additional calendar fields such as Year, Month, Hour, and Weekday Name, while weekly and monthly datasets contain only the date and medicine category columns. The `datum` column is currently stored as an object type and should be converted to datetime during cleaning.


## 6. Missing Values and Duplicate Records

Missing and duplicate records can distort totals and comparisons. This check is performed before any transformations so it describes the raw files exactly as received.

In [24]:
quality_summary = pd.DataFrame([
    {
        "Dataset": name,
        "Missing Values": int(df.isnull().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum()),
    }
    for name, df in datasets.items()
])

quality_summary

,Dataset,Missing Values,Duplicate Rows
0,Daily,0,0
1,Hourly,0,0
2,Weekly,0,0
3,Monthly,0,0


*Note..* The raw datasets do not contain missing values or fully duplicated rows, so no missing-value imputation or duplicate-row removal is required at this stage.


## 7. Date Coverage

The `datum` field is converted to a temporary Series only to inspect coverage and parsing quality. The original DataFrames remain unchanged.

In [25]:
date_data = []

for name, df in datasets.items():
    inspected_dates = pd.to_datetime(df["datum"], errors="coerce")

    raw_type = df["datum"].dtype
    start_date = inspected_dates.min()
    end_date = inspected_dates.max()
    unique_dates = df["datum"].nunique()
    invalid_dates = inspected_dates.isna().sum()

    date_data.append({
        "Dataset": name,
        "Raw Data Type": raw_type,
        "Start": start_date,
        "End": end_date,
        "Unique Values": unique_dates,
        "Unparsed Values": invalid_dates
    })

date_summary = pd.DataFrame(date_data)
date_summary

,Dataset,Raw Data Type,Start,End,Unique Values,Unparsed Values
0,Daily,str,2014-01-02 00:00:00,2019-10-08 00:00:00,2106,0
1,Hourly,str,2014-01-02 08:00:00,2019-10-08 19:00:00,50532,0
2,Weekly,str,2014-01-05 00:00:00,2019-10-13 00:00:00,302,0
3,Monthly,str,2014-01-31 00:00:00,2019-10-31 00:00:00,70,0


**Verified result:** The files cover January 2014 through October 2019, with no date values failing the inspection conversion. Exact start and end timestamps differ by time grain.


*Note..*
Although all `datum` values can be parsed successfully, the column is still stored as a string, so it needs permanent datetime conversion during cleaning to support accurate sorting, filtering, time-based feature creation, and time series analysis.


## 8. Medicine Categories

The common non-date fields identify eight medicine or product groups: `M01AB`, `M01AE`, `N02BA`, `N02BE`, `N05B`, `N05C`, `R03`, and `R06`. Their codes are retained because the source data does not provide product-level names.

In [ ]:
medicine_category_columns = [
    "M01AB", "M01AE", "N02BA", "N02BE",
    "N05B", "N05C", "R03", "R06"
]#as we already know the columns are same for all datasets..

category_types = pd.DataFrame()

for name, df in datasets.items():
    category_types[name] = df[medicine_category_columns].dtypes.astype(str)

category_types

,Daily,Hourly,Weekly,Monthly
M01AB,float64,float64,float64,float64
M01AE,float64,float64,float64,float64
N02BA,float64,float64,float64,float64
N02BE,float64,float64,float64,float64
N05B,float64,float64,float64,float64
N05C,float64,float64,float64,float64
R03,float64,float64,float64,float64
R06,float64,float64,float64,float64


*Note..*
The eight product-group codes — `M01AB`, `M01AE`, `N02BA`, `N02BE`, `N05B`, `N05C`, `R03`, and `R06` — are present and numeric in all four datasets, so the analysis will be performed at the product-group level. Individual medicine names, unit prices, and revenue values are not available in the source data.


## 9. Statistical Sanity Check

A descriptive summary is used here only to spot values that may need validation during cleaning. Business comparisons and category-performance conclusions are intentionally deferred to the EDA notebook.

In [31]:
for name, df in datasets.items():
    category_summary = df[medicine_category_columns].describe().T
    
    print(f"{name} Dataset")
    
    display(category_summary)

Daily Dataset


,count,mean,std,min,25%,50%,75%,max
M01AB,2106.0,5.033683,2.737579,0.0,3.00,4.99,6.670,17.340000
M01AE,2106.0,3.895830,2.133337,0.0,2.34,3.67,5.138,14.463000
N02BA,2106.0,3.880441,2.384010,0.0,2.00,3.50,5.200,16.000000
N02BE,2106.0,29.917095,15.590966,0.0,19.00,26.90,38.300,161.000000
N05B,2106.0,8.853627,5.605605,0.0,5.00,8.00,12.000,54.833333
N05C,2106.0,0.593522,1.092988,0.0,0.00,0.00,1.000,9.000000
R03,2106.0,5.512262,6.428736,0.0,1.00,4.00,8.000,45.000000
R06,2106.0,2.900198,2.415816,0.0,1.00,2.00,4.000,15.000000


Hourly Dataset


,count,mean,std,min,25%,50%,75%,max
M01AB,50532.0,0.209787,0.556003,0.0,0.0,0.0,0.000,7.0
M01AE,50532.0,0.162365,0.416109,0.0,0.0,0.0,0.000,6.0
N02BA,50532.0,0.161723,0.453211,0.0,0.0,0.0,0.000,6.5
N02BE,50532.0,1.246842,2.387392,0.0,0.0,0.0,1.875,29.0
N05B,50532.0,0.368989,0.930934,0.0,0.0,0.0,0.000,15.0
N05C,50532.0,0.024736,0.217871,0.0,0.0,0.0,0.000,6.0
R03,50532.0,0.229732,1.240513,0.0,0.0,0.0,0.000,25.0
R06,50532.0,0.120870,0.391999,0.0,0.0,0.0,0.000,5.0


Weekly Dataset


,count,mean,std,min,25%,50%,75%,max
M01AB,302.0,35.102441,8.617106,7.670,29.3875,34.565000,40.1750,65.330
M01AE,302.0,27.167611,7.043491,6.237,22.3875,26.789500,31.0465,53.571
N02BA,302.0,27.060295,8.086458,3.500,21.3000,26.500000,32.4750,60.125
N02BE,302.0,208.627161,76.069221,86.250,149.3000,198.300000,252.4715,546.899
N05B,302.0,61.740853,22.436970,18.000,47.0000,57.000000,71.0000,154.000
N05C,302.0,4.138935,3.129265,0.000,2.0000,3.979167,6.0000,17.000
R03,302.0,38.439811,22.900873,2.000,21.0000,35.000000,51.0000,131.000
R06,302.0,20.224561,11.381464,1.000,11.4750,17.500000,26.0000,65.000


Monthly Dataset


,count,mean,std,min,25%,50%,75%,max
M01AB,70.0,149.992000,31.485325,0.0,137.49000,154.6350,169.00000,211.130
M01AE,70.0,116.514286,27.889336,0.0,103.51825,114.8400,128.35975,222.351
N02BA,70.0,115.020843,31.245899,0.0,94.37500,117.2250,133.83750,191.600
N02BE,70.0,892.542071,338.843908,0.0,648.18750,865.8245,1061.58000,1856.815
N05B,70.0,262.118571,85.060930,1.0,223.75000,250.3000,293.65000,492.000
N05C,70.0,17.842857,8.481242,0.0,12.00000,18.0000,23.00000,50.000
R03,70.0,167.675000,81.767979,0.0,112.00000,160.0000,218.25000,386.000
R06,70.0,86.662571,45.859336,0.0,49.87500,74.1000,119.80750,213.040


**What I found..**
The medicine-category columns contain valid non-negative numeric values. Monthly records with `0` values in multiple medicine categories should be reviewed during cleaning, and unusually high maximum values such as `N02BE = 161` in daily data and `N02BE = 1856.815` in monthly data should be validated.

**Values to validate:** Monthly zero values across medicine categories and unusually high `N02BE` maximum values.

## 10. Data Understanding Summary

* The project contains four source files at hourly, daily, weekly, and monthly time grains.
* The raw data contains 50,532 hourly records, 2,106 daily records, 302 weekly records, and 70 monthly records.
* `datum` is the main date/time field and needs permanent datetime conversion during cleaning.
* Eight numeric columns represent medicine product groups: `M01AB`, `M01AE`, `N02BA`, `N02BE`, `N05B`, `N05C`, `R03`, and `R06`.
* No missing values or fully duplicated rows were found in the raw datasets.
* The data covers the period from January 2014 to October 2019.
* The data is structurally ready for cleaning, but date types, naming consistency, calendar fields, and calculated total-sales columns still need to be standardized.
